# ⚙️ Notebook 3 — Preprocessing & Feature Engineering
**Project:** FAANG Stock Price Forecasting
**Goal:** Add useful features (moving averages, RSI, lags) and split data into train/test sets for each stock.

## Step 1 — Import Libraries

In [1]:
import pandas as pd
import numpy as np
import os

print('✅ Libraries ready')

✅ Libraries ready


## Step 2 — Load Raw Data for All 5 Stocks

In [2]:
TICKERS = ["META", "AAPL", "AMZN", "NFLX", "GOOGL"]

stock_data = {}
for ticker in TICKERS:
    df = pd.read_csv(f"data/{ticker}_raw.csv", index_col="Date", parse_dates=True)
    stock_data[ticker] = df
    print(f"{ticker}: {df.shape}")

META: (1509, 6)
AAPL: (1509, 6)
AMZN: (1509, 6)
NFLX: (1509, 6)
GOOGL: (1509, 6)


## Step 3 — Feature Engineering (Loop Over All Stocks)

For each stock we add:
- **MA7, MA21** — 7-day & 21-day moving averages
- **Daily Return** — % change from previous day
- **Volatility** — 21-day rolling std of returns
- **RSI (14)** — Relative Strength Index (momentum indicator)
- **Lag_1, Lag_2, Lag_3** — previous days' closing prices

In [3]:
def add_features(df):
    df = df.copy()

    # Moving averages
    df["MA7"]  = df["Close"].rolling(7).mean()
    df["MA21"] = df["Close"].rolling(21).mean()

    # Daily return & volatility
    df["Daily_Return"] = df["Close"].pct_change()
    df["Volatility"]   = df["Daily_Return"].rolling(21).std()

    # RSI (14-day)
    delta = df["Close"].diff()
    gain  = delta.clip(lower=0).rolling(14).mean()
    loss  = (-delta.clip(upper=0)).rolling(14).mean()
    rs    = gain / loss
    df["RSI"] = 100 - (100 / (1 + rs))

    # Lag features
    for lag in [1, 2, 3]:
        df[f"Lag_{lag}"] = df["Close"].shift(lag)

    return df

featured_data = {}
for ticker, df in stock_data.items():
    featured_data[ticker] = add_features(df)

print("✅ Features added for all stocks")
print("New columns:", [c for c in featured_data["AAPL"].columns if c not in stock_data["AAPL"].columns])

✅ Features added for all stocks
New columns: ['MA7', 'MA21', 'Daily_Return', 'Volatility', 'RSI', 'Lag_1', 'Lag_2', 'Lag_3']


## Step 4 — Drop Rows with Missing Values

Rolling averages and lags create `NaN` in the first few rows (not enough history yet) — we drop those.

In [4]:
for ticker in TICKERS:
    before = len(featured_data[ticker])
    featured_data[ticker] = featured_data[ticker].dropna()
    after = len(featured_data[ticker])
    print(f"{ticker}: {before} → {after} rows (dropped {before - after})")

META: 1509 → 1488 rows (dropped 21)
AAPL: 1509 → 1488 rows (dropped 21)
AMZN: 1509 → 1488 rows (dropped 21)
NFLX: 1509 → 1488 rows (dropped 21)
GOOGL: 1509 → 1488 rows (dropped 21)


## Step 5 — Train / Test Split (80/20, Time-Based)

We split by **date order**, not randomly — the model should never "see the future" during training.

In [5]:
train_data = {}
test_data  = {}

for ticker, df in featured_data.items():
    split_idx = int(len(df) * 0.80)
    train_data[ticker] = df.iloc[:split_idx]
    test_data[ticker]  = df.iloc[split_idx:]

    print(f"{ticker}: train {train_data[ticker].index.min().date()} → {train_data[ticker].index.max().date()} "
          f"({len(train_data[ticker])} rows) | "
          f"test {test_data[ticker].index.min().date()} → {test_data[ticker].index.max().date()} "
          f"({len(test_data[ticker])} rows)")

META: train 2019-02-01 → 2023-10-23 (1190 rows) | test 2023-10-24 → 2024-12-30 (298 rows)
AAPL: train 2019-02-01 → 2023-10-23 (1190 rows) | test 2023-10-24 → 2024-12-30 (298 rows)
AMZN: train 2019-02-01 → 2023-10-23 (1190 rows) | test 2023-10-24 → 2024-12-30 (298 rows)
NFLX: train 2019-02-01 → 2023-10-23 (1190 rows) | test 2023-10-24 → 2024-12-30 (298 rows)
GOOGL: train 2019-02-01 → 2023-10-23 (1190 rows) | test 2023-10-24 → 2024-12-30 (298 rows)


## Step 6 — Save Processed Data

In [6]:
os.makedirs("data/processed", exist_ok=True)

for ticker in TICKERS:
    train_data[ticker].to_csv(f"data/processed/{ticker}_train.csv")
    test_data[ticker].to_csv(f"data/processed/{ticker}_test.csv")

print(f"✅ Saved train & test files for {len(TICKERS)} stocks → data/processed/")

✅ Saved train & test files for 5 stocks → data/processed/


## ✅ Notebook 3 Summary

**Features added to each stock:**
- `MA7`, `MA21` — moving averages
- `Daily_Return`, `Volatility` — return & risk
- `RSI` — momentum indicator (0–100)
- `Lag_1`, `Lag_2`, `Lag_3` — previous days' prices

**Train/Test split:** 80% / 20% — split by time (no shuffling)

**Files saved:** `data/processed/{TICKER}_train.csv` and `data/processed/{TICKER}_test.csv` for all 5 stocks

**Next →** `04_modeling.ipynb`